In [1]:
from importlib.metadata import version

print("torch version:", version("torch"))

torch version: 2.8.0


In [2]:
import torch

inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]] # step     (x^6)
)

d:\Self study\Deep Learning\LLM from Scratch\.venv\Lib\site-packages\torch\_subclasses\functional_tensor.py:279: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:81.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


In [3]:
input_query =  inputs[1]
input_query

tensor([0.5500, 0.8700, 0.6600])

#### Attention Scores

In [4]:
# empty tensors
att_scores = torch.empty(inputs.shape[0])

for i , emb in enumerate(inputs):
    att_scores[i] = torch.dot(emb , input_query)


att_scores
    

tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])

#### Attention Weights

In [5]:
# get the normalization using softmax
def softmax_naive(x):
    return torch.exp(x) / torch.exp(x).sum()    

softmax_naive(att_scores)

tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])

In [6]:
att_normalized =  torch.softmax(att_scores , dim=0)

In [7]:
# context vector = sum(attention_weights * embeddings)
ctx_vector = torch.sum(att_normalized.unsqueeze(1) * inputs, dim=0)


In [8]:
ctx_vector

tensor([0.4419, 0.6515, 0.5683])

Computing attention weights for all input tokens

In [9]:
inputs.shape

torch.Size([6, 3])

In [10]:
# empty tensors
att_scores = torch.empty(6,6)

for i , x_i in enumerate(inputs):
    for j , x_j in enumerate(inputs):
        att_scores[i,j] = torch.dot(x_i , x_j)


att_scores

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])

In [11]:
att_scores = inputs @ inputs.T
att_scores

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])

In [12]:
atn_weights = torch.softmax(att_scores , dim =1)
atn_weights

tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])

In [13]:
all_ctx_vecs =  atn_weights @ inputs
all_ctx_vecs

tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])

#### Attention Mechanism with trainable parameters

In [ ]:
inputs

tensor([[0.4300, 0.1500, 0.8900],
        [0.5500, 0.8700, 0.6600],
        [0.5700, 0.8500, 0.6400],
        [0.2200, 0.5800, 0.3300],
        [0.7700, 0.2500, 0.1000],
        [0.0500, 0.8000, 0.5500]])

In [15]:
x_2 =  inputs[1]
d_in = inputs.shape[1]  # here is the input embedding size
d_out = 2

In [ ]:
torch.manual_seed(123)

# create the query weight matrix
W_query =torch.nn.Parameter(torch.rand(d_in , d_out))
W_key =torch.nn.Parameter(torch.rand(d_in , d_out))
W_value =torch.nn.Parameter(torch.rand(d_in , d_out))

print(W_query, W_key , W_value)

Parameter containing:
tensor([[0.2961, 0.5166],
        [0.2517, 0.6886],
        [0.0740, 0.8665]], requires_grad=True) Parameter containing:
tensor([[0.1366, 0.1025],
        [0.1841, 0.7264],
        [0.3153, 0.6871]], requires_grad=True) Parameter containing:
tensor([[0.0756, 0.1966],
        [0.3164, 0.4017],
        [0.1186, 0.8274]], requires_grad=True)


In [17]:
# printing shapes for references
print(x_2)
print(x_2.shape , W_query.shape)

query_2 = x_2 @ W_query

print(query_2.shape , query_2)

tensor([0.5500, 0.8700, 0.6600])
torch.Size([3]) torch.Size([3, 2])
torch.Size([2]) tensor([0.4306, 1.4551], grad_fn=<SqueezeBackward4>)


In [18]:
# Here query for the considered token is the same and we need to calculate the keys and values for each token

# * Here keep in mide the query key and val are random weights and will be trained later
keys = inputs @ W_key
values = inputs @ W_value

keys , values

(tensor([[0.3669, 0.7646],
         [0.4433, 1.1419],
         [0.4361, 1.1156],
         [0.2408, 0.6706],
         [0.1827, 0.3292],
         [0.3275, 0.9642]], grad_fn=<MmBackward0>),
 tensor([[0.1855, 0.8812],
         [0.3951, 1.0037],
         [0.3879, 0.9831],
         [0.2393, 0.5493],
         [0.1492, 0.3346],
         [0.3221, 0.7863]], grad_fn=<MmBackward0>))

![image](./images/1.png)

In [19]:
keys_2 = keys[1]
att_score_22 = torch.dot(query_2 , keys_2)

att_score_22

tensor(1.8524, grad_fn=<DotBackward0>)

In [20]:
# Here we can calculate all at onece with the query2 (see the image above)

attns_scores_2 = query_2 @ keys.T

attns_scores_2

tensor([1.2705, 1.8524, 1.8111, 1.0795, 0.5577, 1.5440],
       grad_fn=<SqueezeBackward4>)

In [21]:
d_k = keys.shape[1]

att_wts_2 = torch.softmax(attns_scores_2 / d_k**0.5 , dim=-1)

torch.sum(att_wts_2)

tensor(1., grad_fn=<SumBackward0>)

![image](./images/2.png)

In [22]:
context_vec_2 = att_wts_2 @ values
print(context_vec_2)

tensor([0.3061, 0.8210], grad_fn=<SqueezeBackward4>)


Now we can ge the full contect vector for all the queries

In [23]:
inputs

tensor([[0.4300, 0.1500, 0.8900],
        [0.5500, 0.8700, 0.6600],
        [0.5700, 0.8500, 0.6400],
        [0.2200, 0.5800, 0.3300],
        [0.7700, 0.2500, 0.1000],
        [0.0500, 0.8000, 0.5500]])

In [24]:
# init the QKV
d_in ,d_out

(3, 2)

In [68]:
torch.manual_seed(123)

W_query =torch.nn.Parameter(torch.rand(d_in , d_out))
W_key =torch.nn.Parameter(torch.rand(d_in , d_out))
W_value =torch.nn.Parameter(torch.rand(d_in , d_out))

In [69]:
inputs.shape , W_query.shape 

(torch.Size([6, 3]), torch.Size([3, 2]))

In [70]:
# All matrixes

query = inputs @ W_query
key = inputs @ W_key
value = inputs @ W_value

In [71]:
d_k

2

In [72]:
class SelfAttention(): 
    def __init__(self , query , key ,value):
        self.query = query
        self.key = key
        self.value = value

    def calContextVec(self):

        # ctx_vector =  torch.empty() 

        att_matrix = self.query @ self.key.T / self.key.shape[-1]**0.5
        att_weights = torch.softmax(att_matrix , dim=-1)
        ctx_vector = att_weights @ self.value

        return ctx_vector
                    

In [73]:
self_attention =  SelfAttention(query , key , value)

In [74]:
ctx_vec = self_attention.calContextVec()

print(ctx_vec)

tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]], grad_fn=<MmBackward0>)


#### Here are the thing that we need to think over my implementation
- Here all of the parameters in my code (q,k,v) are already defined so they are static and wont be trainable
- Here I have not defined the faward methode() this is a pytorch api wher that is optimized from optimizer.step() methode
- Not following the PyTorch best practice

In [85]:
# Correct implementation
import torch.nn as nn

class SelfAttention_v1(nn.Module):
    def __init__(self , d_in , d_out):
        super().__init__() # Call the __init__() method of the parent class (nn.Module) before doing anything else
        self.W_query = nn.Parameter(torch.rand(d_in, d_out))
        self.W_key   = nn.Parameter(torch.rand(d_in, d_out))
        self.W_value = nn.Parameter(torch.rand(d_in, d_out))

    
    def forward(self ,x):

        keys =  x @ self.W_key
        values =  x @ self.W_value
        queries =  x @ self.W_query

        attn_scores = queries @ keys.T # omega
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5, dim=-1
        )

        context_vec = attn_weights @ values
        return context_vec


In [86]:
torch.manual_seed(123)
sa_v1 = SelfAttention_v1(d_in, d_out)
print(sa_v1(inputs))

tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]], grad_fn=<MmBackward0>)


In [87]:
# Here is a more optimal version from using Linear Layers
class SelfAttention_v2(nn.Module):
    def __init__(self , d_in ,d_out ,qkv_bias=False):
        super().__init__()

        self.W_query = nn.Linear(d_in, d_out , bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out , bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out , bias=qkv_bias)

    def forward(self , x):

        queries = self.W_query(x)
        keys = self.W_key (x)
        values = self.W_value(x)

        attn_scores = queries @ keys.T # omega
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5, dim=-1
        )

        context_vec = attn_weights @ values
        return context_vec




In [93]:
self_att2 =  SelfAttention_v2(d_in , d_out , False)
ctx_vector_2 = self_att2(inputs)

ctx_vector_2

tensor([[-0.2410,  0.2378],
        [-0.2419,  0.2406],
        [-0.2418,  0.2403],
        [-0.2409,  0.2409],
        [-0.2392,  0.2355],
        [-0.2420,  0.2433]], grad_fn=<MmBackward0>)

#### Hiding future words with causal attention

![causal_attention](./images//3.png)

In [148]:
queries = self_att2.W_query(inputs)
keys = self_att2.W_key (inputs)
values = self_att2.W_value(inputs)

attn_scores = queries @ keys.T # omega
attn_weights  = torch.softmax(
attn_scores / keys.shape[-1]**0.5, dim=-1
)

In [149]:
attn_weights 

tensor([[0.1566, 0.1692, 0.1691, 0.1690, 0.1662, 0.1699],
        [0.1623, 0.1748, 0.1746, 0.1614, 0.1624, 0.1644],
        [0.1619, 0.1743, 0.1741, 0.1621, 0.1628, 0.1649],
        [0.1667, 0.1729, 0.1728, 0.1611, 0.1631, 0.1634],
        [0.1560, 0.1622, 0.1623, 0.1756, 0.1702, 0.1738],
        [0.1703, 0.1784, 0.1781, 0.1547, 0.1595, 0.1589]],
       grad_fn=<SoftmaxBackward0>)

In [150]:
masked_att = torch.tril(attn_weights )

In [143]:
masked_att

tensor([[0.1566, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1623, 0.1748, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1619, 0.1743, 0.1741, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1729, 0.1728, 0.1611, 0.0000, 0.0000],
        [0.1560, 0.1622, 0.1623, 0.1756, 0.1702, 0.0000],
        [0.1703, 0.1784, 0.1781, 0.1547, 0.1595, 0.1589]],
       grad_fn=<TrilBackward0>)

In [151]:
# as the row sum is not 1 we need to normalize this again

#? impl -1

# mask = torch.tril(torch.ones_like(attn_scores))  # lower-triangular mask
# masked_att = attn_scores.masked_fill(mask == 0, float('-inf'))
# attn_weights = torch.softmax(masked_att, dim=-1)
# attn_weights

#? impl -2

row_sum =  masked_att.sum(dim=-1 , keepdim=True)
masked_att_norm =  masked_att / row_sum

masked_att_norm


tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4814, 0.5186, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3173, 0.3416, 0.3412, 0.0000, 0.0000, 0.0000],
        [0.2475, 0.2567, 0.2565, 0.2392, 0.0000, 0.0000],
        [0.1887, 0.1964, 0.1965, 0.2125, 0.2059, 0.0000],
        [0.1703, 0.1784, 0.1781, 0.1547, 0.1595, 0.1589]],
       grad_fn=<DivBackward0>)

In [152]:
# we can do the same with only 3 steps
# current way= att_scores -> normalize ->causal att ->re normalized

# we can do = att_scores -> causal_att -> normalize 

context_length = attn_scores.shape[0]

mask = torch.triu(torch.ones(context_length, context_length), diagonal=1)
masked = attn_scores.masked_fill(mask.bool(), -torch.inf)
print(masked)

tensor([[-0.0691,    -inf,    -inf,    -inf,    -inf,    -inf],
        [ 0.1479,  0.2531,    -inf,    -inf,    -inf,    -inf],
        [ 0.1287,  0.2329,  0.2313,    -inf,    -inf,    -inf],
        [ 0.1587,  0.2104,  0.2091,  0.1105,    -inf,    -inf],
        [-0.2529, -0.1970, -0.1963, -0.0853, -0.1296,    -inf],
        [ 0.3459,  0.4120,  0.4097,  0.2103,  0.2532,  0.2483]],
       grad_fn=<MaskedFillBackward0>)


In [153]:
attn_weights = torch.softmax(masked / keys.shape[-1]**0.5, dim=-1)
print(attn_weights)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4814, 0.5186, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3173, 0.3416, 0.3412, 0.0000, 0.0000, 0.0000],
        [0.2475, 0.2567, 0.2565, 0.2392, 0.0000, 0.0000],
        [0.1887, 0.1964, 0.1965, 0.2125, 0.2059, 0.0000],
        [0.1703, 0.1784, 0.1781, 0.1547, 0.1595, 0.1589]],
       grad_fn=<SoftmaxBackward0>)


Masking additional attention weights with dropout

![dropout](./images/4.png)